<a href="https://colab.research.google.com/github/naubergois/curso-transfer-colab/blob/main/exercicios/aula2-exercicio6-pesos-do-bloco.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# Aula 2, exercício 6: Pesos do bloco

Quantos parâmetros tem o `layer4`, livres e parados somados?

O exercício é da aula [2. Congelar e soltar](https://colab.research.google.com/github/naubergois/curso-transfer-colab/blob/main/notebooks/02-congelar-e-soltar.ipynb), e este notebook roda sozinho. Use **Ambiente de execução → Executar tudo**: ele prepara o modelo, deixa uma célula para você tentar e, no fim, mostra uma solução já conferida.

In [ ]:
!pip install -q torch torchvision

## Preparação

A preparação congela a base, troca a cabeça e solta só o `layer4`. O otimizador fica com duas taxas: um passo curto na base, um passo maior na cabeça.

In [ ]:
def contar(modelo, so_treinavel=False):
    total = 0
    for p in modelo.parameters():
        if so_treinavel and not p.requires_grad:
            continue
        total += p.numel()
    return total


import torch
from torch import nn
from torchvision.models import ResNet18_Weights, resnet18

torch.manual_seed(7)

modelo = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
modelo.eval()
entrada = modelo.fc.in_features
for p in modelo.parameters():
    p.requires_grad = False
modelo.fc = nn.Linear(entrada, 2)
for p in modelo.layer4.parameters():
    p.requires_grad = True

grupo_base = [p for p in modelo.layer4.parameters() if p.requires_grad]
grupo_cabeca = list(modelo.fc.parameters())
otimizador = torch.optim.Adam([
    {"params": grupo_base, "lr": 1e-4},
    {"params": grupo_cabeca, "lr": 1e-3},
])

print(f"conv1 pode mudar? {modelo.conv1.weight.requires_grad}")
print(f"layer4 pode mudar? {all(p.requires_grad for p in modelo.layer4.parameters())}")
print(f"grupos de taxa: {len(otimizador.param_groups)}")
print(f"passo da base: {otimizador.param_groups[0]['lr']}")
print(f"passo da cabeça: {otimizador.param_groups[1]['lr']}")
print(f"pesos do layer4: {contar(modelo.layer4)}")

In [ ]:
# @title Correção automática (rode, não precisa ler)
VERIFICAR = {
    1: lambda r: bool(r) is False,
    2: lambda r: bool(r) is True,
    3: lambda r: int(r) == len(otimizador.param_groups),
    4: lambda r: abs(float(r) - otimizador.param_groups[0]["lr"]) < 1e-12,
    5: lambda r: abs(float(r) - otimizador.param_groups[1]["lr"]) < 1e-12,
    6: lambda r: int(r) == contar(modelo.layer4),
}

_acertos = {}


def conferir(n, resposta):
    if resposta is None:
        print(f"Exercício {n}: ainda sem resposta.")
        return
    try:
        certo = bool(VERIFICAR[n](resposta))
    except Exception as erro:
        certo = False
        print(f"  (a verificação deu erro: {erro})")
    _acertos[n] = certo
    print(f"Exercício {n}: " + ("✅ certo!" if certo else "❌ ainda não. Releia o enunciado e tente de novo."))


def placar():
    print(f"{sum(_acertos.values())} de {len(VERIFICAR)} exercícios certos.")

## Sua vez

Quantos parâmetros tem o `layer4`, livres e parados somados?

*Dica:* contar(modelo.layer4)

Troque o `None` pela sua resposta e rode a célula.

In [ ]:
resposta = None

conferir(6, resposta)

## Uma solução

Compare com o que você fez. Existem outros caminhos certos: o que vale é a `conferir` aprovar.

In [ ]:
resposta = contar(modelo.layer4)

conferir(6, resposta)

## O que fica

Próximo: [Aula 3, exercício 1: Quantas fotos de treino](https://colab.research.google.com/github/naubergois/curso-transfer-colab/blob/main/exercicios/aula3-exercicio1-quantas-fotos-de-treino.ipynb).